<a href="https://colab.research.google.com/github/arun-antony/ai-colab-tryout/blob/main/351860_non_llm_grading_assistance_notebook_v2_mini_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Import dependencies


In [ ]:
from pathlib import Path
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## 2. Configure project paths

Keep the notebook in the root project folder, next to the `data/` and `outputs/` folders.


In [ ]:
PROJECT_DIR = Path(".")
DATA_DIR = PROJECT_DIR / "data"
SUBMISSIONS_DIR = DATA_DIR / "submissions"
OUTPUTS_DIR = PROJECT_DIR / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

RUBRIC_PATH = DATA_DIR / "ai_ml_dl_rubric.xlsx"
REFERENCE_PATH = DATA_DIR / "reference_answers_ai_ml_dl.xlsx"
ASSIGNMENT_PATH = DATA_DIR / "assignment_sheet_ai_ml_dl.xlsx"
MAPPING_PATH = DATA_DIR / "evaluation_mapping_document.xlsx"
VALIDATION_PATH = DATA_DIR / "validation_dataset.xlsx"

required_files = [RUBRIC_PATH, REFERENCE_PATH, ASSIGNMENT_PATH, MAPPING_PATH, VALIDATION_PATH]
missing = [str(p) for p in required_files if not p.exists()]

if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

print("All required project files found.")


## 3. Load project documents


In [ ]:
rubric_df = pd.read_excel(RUBRIC_PATH)
reference_df = pd.read_excel(REFERENCE_PATH)
assignment_df = pd.read_excel(ASSIGNMENT_PATH, header=None)
mapping_df = pd.read_excel(MAPPING_PATH)
validation_df = pd.read_excel(VALIDATION_PATH)

print("Rubric rows:", len(rubric_df))
print("Reference answer rows:", len(reference_df))
print("Mapping rows:", len(mapping_df))
print("Validation rows:", len(validation_df))

rubric_df.head()


## 4. Normalize and validate document structure

The notebook expects `question_id` to be the shared key across rubric, reference answers, mapping, and submissions.


In [ ]:
def normalize_question_id(value):
    match = re.search(r"Q\s*(\d+)", str(value), flags=re.IGNORECASE)
    return f"Q{int(match.group(1))}" if match else str(value).strip()

for df in [rubric_df, reference_df, mapping_df, validation_df]:
    if "question_id" in df.columns:
        df["question_id"] = df["question_id"].apply(normalize_question_id)

expected_qids = {f"Q{i}" for i in range(1, 21)}

rubric_qids = set(rubric_df["question_id"])
reference_qids = set(reference_df["question_id"])
mapping_qids = set(mapping_df["question_id"])

print("Rubric complete:", expected_qids.issubset(rubric_qids))
print("Reference answers complete:", expected_qids.issubset(reference_qids))
print("Mapping complete:", expected_qids.issubset(mapping_qids))


## 5. Load student submissions

Place student submissions in `data/submissions/`.

Recommended plain-text format:

```text
Student ID: student_001

Q1:
Student answer...

Q2:
Student answer...
```

The project ZIP includes one sample submission so you can test the notebook immediately.


In [ ]:
def read_text_file(path: Path) -> str:
    return path.read_text(encoding="utf-8")

def parse_submission(text: str):
    student_id_match = re.search(r"Student\s*ID\s*:\s*(.+)", text, flags=re.IGNORECASE)
    student_id = student_id_match.group(1).strip() if student_id_match else "unknown_student"

    pattern = r"(Q\s*\d+)\s*:\s*(.*?)(?=\n\s*Q\s*\d+\s*:|\Z)"
    matches = re.findall(pattern, text, flags=re.IGNORECASE | re.DOTALL)

    rows = []
    for qid, answer in matches:
        rows.append({
            "student_id": student_id,
            "question_id": normalize_question_id(qid),
            "student_answer": re.sub(r"\s+", " ", answer).strip()
        })

    return rows

submission_rows = []

for path in sorted(SUBMISSIONS_DIR.glob("*.txt")):
    submission_rows.extend(parse_submission(read_text_file(path)))

submissions_df = pd.DataFrame(submission_rows)

if submissions_df.empty:
    raise ValueError("No valid submissions found in data/submissions/. Add at least one .txt file.")

print("Loaded student answer rows:", len(submissions_df))
submissions_df.head()


## 6. Build a per-question retrieval corpus

For each question, the retriever uses only the matching rubric, reference answer, and mapping row. This prevents Q6 answers from being compared against Q13 or Q17 by mistake.


In [ ]:
def row_to_text(row):
    parts = []
    for col, value in row.items():
        if pd.notna(value):
            parts.append(f"{col}: {value}")
    return " | ".join(parts)

knowledge_rows = []

for _, row in rubric_df.iterrows():
    knowledge_rows.append({
        "question_id": row["question_id"],
        "source": "rubric",
        "text": row_to_text(row)
    })

for _, row in reference_df.iterrows():
    knowledge_rows.append({
        "question_id": row["question_id"],
        "source": "reference_answer",
        "text": row_to_text(row)
    })

for _, row in mapping_df.iterrows():
    knowledge_rows.append({
        "question_id": row["question_id"],
        "source": "evaluation_mapping",
        "text": row_to_text(row)
    })

knowledge_df = pd.DataFrame(knowledge_rows)
knowledge_df.head()


## 7. Helper functions for TF-IDF retrieval and scoring


In [ ]:
def cosine_tfidf_similarity(text_a, text_b):
    vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
    matrix = vectorizer.fit_transform([str(text_a), str(text_b)])
    return float(cosine_similarity(matrix[0], matrix[1])[0][0])

def split_terms(value):
    if pd.isna(value):
        return []
    return [t.strip().lower() for t in str(value).split("|") if t.strip()]

def concept_coverage(answer, required_terms):
    answer_lower = str(answer).lower()
    terms = split_terms(required_terms)
    if not terms:
        return 0.0, []

    matched = [term for term in terms if term in answer_lower]
    return len(matched) / len(terms), matched

def misconception_hits(answer, misconceptions):
    answer_lower = str(answer).lower()
    terms = split_terms(misconceptions)
    return [term for term in terms if term in answer_lower]

def get_single_row(df, question_id):
    rows = df[df["question_id"] == question_id]
    if rows.empty:
        return None
    return rows.iloc[0]


### 7.1 Answer Length Validation

In [ ]:
ANSWER_TOO_SHORT_THRESHOLD = 10
ANSWER_TOO_LONG_THRESHOLD = 300

def get_num_words(answer):
    words = re.findall(r'\w+', str(answer))
    return len(words)

# This Guardrail Function wil try to see if an answer is too long
# or too short (even empty answer submission) based on threshold.
# This will check for number of words in the answer to determine validity.
def is_answer_length_valid(answer):
    num_words = get_num_words(answer)
    if num_words < ANSWER_TOO_SHORT_THRESHOLD:
        return False, f"Answer is too short ({num_words} words)"
    elif num_words > ANSWER_TOO_LONG_THRESHOLD:
        return False, f"Answer is too long ({num_words} words)"
    else:
        return True, "Answer length is valid"

### 7.2 Misconception Penalty

In [ ]:
# Misconception penalty as a percentage
MISCONCEPTION_PENALTY = 0.20 # Idea is that if there are 5 misconceptions, the score is ZERO
def get_misconception_penalty(misconceptions_list):
    return MISCONCEPTION_PENALTY * len(misconception_list)

### 7.3 Outlier Detection

In [ ]:
CONCEPT_COVERAGE_MIN_THRESHOLD = 0.30
CONCEPT_COVERAGE_MAX_THRESHOLD = 0.60
SIMILARITY_MIN_THRESHOLD = 0.30
SIMILARITY_MAX_THRESHOLD = 0.60
# This Guardrail function will detect if an answer can be considered an outlier.
# 1. If similarity is very high (> 60%), but concept coverage is low (< 30%), it is considered an outlier
# 2. If similarity is very low (< 30%), but concept coverage is high (> 60%), it is considered an outlier
# 3. Very long answer and poor coverage
def is_considered_outlier(answer, similarity, coverage):
    if similarity >= SIMILARITY_MAX_THRESHOLD and coverage <= CONCEPT_COVERAGE_MIN_THRESHOLD:
        return True, "High similarity, Low coverage"
    elif similarity < SIMILARITY_MIN_THRESHOLD and coverage >= CONCEPT_COVERAGE_MAX_THRESHOLD:
        return True, "Low similarity, High coverage"
    elif get_num_words(answer) > ANSWER_TOO_LONG_THRESHOLD and coverage <= CONCEPT_COVERAGE_MIN_THRESHOLD:
        return True, "Long answer, Low coverage"
    else:
        return False, ""

## 8. Score submissions

The suggested score is a simple deterministic MVP score:

- 70% concept coverage
- 30% TF-IDF similarity to the reference answer
- capped by `max_points`
- final grade still requires instructor review


In [ ]:
review_rows = []

for _, submission in submissions_df.iterrows():
    qid = submission["question_id"]
    answer = submission["student_answer"]

    instructor_notes_list = []

    # Guardrail 1: Answer Length Validation
    is_length_valid, length_reason = is_answer_length_valid(answer)
    if not is_length_valid:
        instructor_notes_list.append(f"Answer Length: {length_reason}")


    rubric_row = get_single_row(rubric_df, qid)
    reference_row = get_single_row(reference_df, qid)
    mapping_row = get_single_row(mapping_df, qid)

    if rubric_row is None or reference_row is None:
        review_rows.append({
            "student_id": submission["student_id"],
            "question_id": qid,
            "student_answer": answer,
            "error": "Missing rubric or reference answer",
        })
        continue

    reference_answer = reference_row.get("ideal_answer", "")
    required_terms = reference_row.get("required_terms", rubric_row.get("accepted_keywords", ""))
    misconceptions = reference_row.get("misconceptions", rubric_row.get("misconceptions", ""))

    similarity = cosine_tfidf_similarity(answer, reference_answer)
    coverage, matched_terms = concept_coverage(answer, required_terms)
    misconception_list = misconception_hits(answer, misconceptions)


    # Guardrail 2: Outlier Detection
    is_outlier, outlier_reason = is_considered_outlier(answer, similarity, coverage)
    if is_outlier:
        instructor_notes_list.append(f"Outlier: {outlier_reason}")

    # Guardrail 3: Misconception Penalty
    misconception_penalty = get_misconception_penalty(misconception_list)
    if misconception_penalty > 0:
        instructor_notes_list.append(f"Misconception Penalty: {misconception_penalty * 100}%")


    max_points = float(rubric_row.get("max_points", 5))
    threshold = float(mapping_row.get("similarity_threshold", 0.35)) if mapping_row is not None else 0.35

    weighted_score_ratio = (0.70 * coverage) + (0.30 * similarity)

    # Subtract penalties if there is misconception
    weighted_score_ratio-= misconception_penalty

    suggested_score = round(weighted_score_ratio * max_points, 2)

    # Clamp the score within range. 0 if it goes negative, max_points if it goes beyond.
    suggested_score = max(0, min(suggested_score, max_points))

    final_decision = ""
    instructor_notes = "\n".join(instructor_notes_list) if instructor_notes_list else ""

    if instructor_notes_list:
      final_decision = "Human Review Required. Details in Instructor Notes."
    else:
      final_decision = "Passed"



    review_rows.append({
        "student_id": submission["student_id"],
        "question_id": qid,
        "student_answer": answer,
        "reference_answer": reference_answer,
        "tfidf_similarity": round(similarity, 3),
        "similarity_threshold": threshold,
        "above_threshold": similarity >= threshold,
        "concept_coverage": round(coverage, 3),
        "matched_terms": "|".join(matched_terms),
        "misconception_detected": "|".join(misconception_list),
        "max_points": max_points,
        "suggested_score": suggested_score,
        "instructor_score": "",
        "instructor_notes": instructor_notes,
        "final_decision": final_decision
    })

review_df = pd.DataFrame(review_rows)
review_df.head(20)


## 9. Export instructor review file


In [ ]:
output_path = OUTPUTS_DIR / "grading_review_output.csv"
review_df.to_csv(output_path, index=False)
print(f"Saved instructor review file to: {output_path}")


## 10. Run a quick validation check

This checks whether the validation examples behave as expected. It is not a full benchmark, but it gives a fast sanity check before grading real submissions.


In [ ]:
validation_results = []

for _, row in validation_df.iterrows():
    qid = row["question_id"]
    answer = row["sample_answer"]

    reference_row = get_single_row(reference_df, qid)
    rubric_row = get_single_row(rubric_df, qid)

    if reference_row is None or rubric_row is None:
        continue

    similarity = cosine_tfidf_similarity(answer, reference_row.get("ideal_answer", ""))
    coverage, matched_terms = concept_coverage(answer, reference_row.get("required_terms", ""))
    score_signal = (0.70 * coverage) + (0.30 * similarity)

    validation_results.append({
        "question_id": qid,
        "answer_quality": row["answer_quality"],
        "sample_answer": answer,
        "tfidf_similarity": round(similarity, 3),
        "concept_coverage": round(coverage, 3),
        "score_signal": round(score_signal, 3),
        "expected_result": row["expected_result"]
    })

validation_results_df = pd.DataFrame(validation_results)
validation_results_df
